In [12]:
import numpy as np
import pandas as pd
from pathlib import Path
import xgboost as xgb
import onnxruntime as ort
import onnx

# ONNX converter for XGBoost (works only with XGBoost <= 1.7.6)
import onnxmltools
from onnxmltools.convert.common.data_types import FloatTensorType

# ---------------------------------------------------------
# Robust project root detection
# ---------------------------------------------------------
def find_project_root(start_path: Path) -> Path:
    for parent in [start_path] + list(start_path.parents):
        if (parent / "data").exists() and (parent / "scripts").exists():
            return parent
    raise RuntimeError("Project root not found.")

PROJECT_ROOT = find_project_root(Path.cwd())

MODEL_DIR = PROJECT_ROOT / "models"
DATA_LABELS_DIR = PROJECT_ROOT / "data" / "labels"

print("Project root:", PROJECT_ROOT)
print("Model dir:", MODEL_DIR)
print("Labels dir:", DATA_LABELS_DIR)


ImportError: cannot import name 'split_complex_to_pairs' from 'onnx.helper' (c:\Trading\Projects\ES_AI_Project\es_env\Lib\site-packages\onnx\helper.py)

In [ ]:
# Load trained XGBoost model
model = xgb.XGBClassifier()
model.load_model(MODEL_DIR / "xgb_model.json")

# Load feature list
with open(MODEL_DIR / "feature_list.txt") as f:
    FEATURE_COLS = [line.strip() for line in f.readlines()]

# Load labeled dataset
df = pd.read_csv(DATA_LABELS_DIR / "labeled.csv")

X = df[FEATURE_COLS].astype(np.float32)

print("Loaded model and dataset.")
print("Feature count:", X.shape[1])


In [ ]:
# Extract raw booster (required for onnxmltools)
booster = model.get_booster()

# Define ONNX input shape
initial_type = [('input', FloatTensorType([None, X.shape[1]]))]

# Convert using onnxmltools (works with XGBoost 1.7.6)
onnx_model = onnxmltools.convert_xgboost(
    booster,
    initial_types=initial_type,
    target_opset=13
)

# Save ONNX model
onnx_path = MODEL_DIR / "xgb_model.onnx"
with open(onnx_path, "wb") as f:
    f.write(onnx_model.SerializeToString())

onnx_path


In [ ]:
# XGBoost predictions
xgb_pred = model.predict_proba(X)[:,1]

# ONNX runtime session
sess = ort.InferenceSession(str(onnx_path))
input_name = sess.get_inputs()[0].name

onnx_pred = sess.run(None, {input_name: X.values.astype(np.float32)})[1].ravel()

# Compare predictions
diff = np.abs(xgb_pred - onnx_pred)

print("Max difference:", diff.max())
print("Mean difference:", diff.mean())
print("ONNX export validation complete.")


In [ ]:
print("ONNX model successfully exported to:")
print(onnx_path)


In [13]:
import lightgbm
import onnx
import onnxruntime
import skl2onnx
